# Kisara — Image-to-Video gratis di Colab (Wan 2.2 TI2V-5B)

Notebook ini mengubah **gambar klip** Kisara menjadi **video** memakai model open-source
Wan 2.2 TI2V-5B, di GPU gratis Colab (T4, 16 GB). Tidak perlu API key dan tidak ada biaya.

**Cara pakai singkat**
1. Menu **Runtime → Change runtime type → T4 GPU**, lalu **Save**.
2. Jalankan sel dari atas ke bawah (Shift+Enter).
3. Di Kisara: buka proyek → langkah **Storyboard** → tombol **Impor video** (atau di **Editor → tab Klip → Impor video**).

> Colab gratis itu **temporary**: sesi mati kalau dibiarkan nganggur atau setelah maksimal 12 jam,
> dan GPU tidak dijamin ada. Karena itu notebook ini dibuat *resumable* — video yang sudah jadi
> akan dilewati, jadi kamu bisa lanjut setelah sesi terputus.
>
> **Catatan aturan:** Colab hanya dipakai sebagai notebook interaktif di sini, bukan sebagai server/backend.
> Itu memang yang diperbolehkan; menjadikannya backend untuk aplikasi lain tidak diperbolehkan.

## 1. Pastikan dapat GPU
Kalau bagian ini tidak menampilkan `Tesla T4`, ubah dulu runtime ke GPU lewat menu Runtime.

In [ ]:
!nvidia-smi

## 2. Pasang dependensi
Wan 2.2 butuh `diffusers` yang cukup baru. Kalau sel berikutnya gagal karena model tidak dikenali,
hapus tanda `#` pada baris `git+https://...` lalu jalankan ulang sel ini.

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate ftfy imageio imageio-ffmpeg
# !pip install -q git+https://github.com/huggingface/diffusers

## 3. Siapkan gambar klip
Urutkan gambar sesuai urutan klip. Dua cara: **mount Google Drive** (paling nyaman untuk banyak gambar)
atau **unggah satu file ZIP** berisi semua gambar.

Gambar disortir berdasarkan nama, jadi beri nama seperti `001.png`, `002.png`, `003.png` supaya urutannya pasti.

In [ ]:
import os, zipfile, shutil
from pathlib import Path

USE_DRIVE = False                                   # True = pakai folder di Google Drive
DRIVE_IMAGES = '/content/drive/MyDrive/kisara/images'
UPLOAD_ZIP = False                                  # True = unggah ZIP gambar dari komputer

IN_DIR = Path(DRIVE_IMAGES)
OUT_DIR = Path('/content/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if UPLOAD_ZIP:
    from google.colab import files
    uploaded = files.upload()                       # pilih satu file .zip berisi gambar
    IN_DIR = Path('/content/images')
    IN_DIR.mkdir(parents=True, exist_ok=True)
    for name in uploaded:
        with zipfile.ZipFile(name) as z:
            z.extractall(IN_DIR)

EXTS = {'.png', '.jpg', '.jpeg', '.webp'}
imgs = sorted(p for p in IN_DIR.rglob('*') if p.suffix.lower() in EXTS)
print(f'{len(imgs)} gambar ditemukan di {IN_DIR}')
for p in imgs[:8]:
    print(' -', p.name)
if len(imgs) > 8:
    print(f'   ... dan {len(imgs) - 8} lagi')

## 4. Atur prompt gerak dan ukuran
Isi `PROMPTS` sejajar dengan urutan gambar (indeks 0 = gambar pertama). Kosongkan (`''`) untuk memakai
`DEFAULT_PROMPT`. Tulis arahan gerak + kamera, seperti kamu memberi instruksi ke kameramen.

**Ukuran:** T4 hanya 16 GB. `480x832` dengan `49` frame (~2 detik) adalah titik aman. Naikkan hanya
kalau VRAM kamu lebih besar; `704x1280` (720p) biasanya kehabisan memori di T4.

In [ ]:
DEFAULT_PROMPT = ('kamera bergerak pelan dan halus, subjek bernafas wajar, gerakan natural, '
                  'sinematik, detail tajam, tanpa potongan gambar')

PROMPTS = [
    # 'Gajah Mada mengangkat kepalan tangan, kamera mendekat pelan',
    # 'asap mengepul pelan di latar belakang, daun bergoyang tertiup angin',
]

NEGATIVE_PROMPT = ('色调艳丽, 过曝, 静态, 细节模糊不清, 字幕, 风格, 作品, 画作, 画面, 静止, 整体发灰, '
                   '最差质量, 低质量, JPEG压缩残留, 丑陋的, 残缺的, 多余的手指, 画得不好的手部, '
                   '画得不好的脸部, 畸形的, 毁容的, 形态畸形的肢体, 手指融合, 静止不动的画面, '
                   '杂乱的背景, 三条腿, 背景人很多, 倒着走')

HEIGHT, WIDTH = 480, 832      # 720p = 704 x 1280 (berat untuk T4)
NUM_FRAMES = 49               # 49 frame @ 24 fps ≈ 2 detik
STEPS = 30
GUIDANCE = 5.0
FPS = 24
print('setelan siap')

## 5. Muat model
Unduhan pertama sekitar 10 GB dan perlu beberapa menit. `enable_model_cpu_offload()` menjaga pemakaian
VRAM tetap rendah supaya muat di kartu 16 GB.

Wan 2.2 dilatih dengan `bfloat16`. T4 (Turing) tidak punya bf16 native, jadi jalannya lebih lambat
tapi hasilnya benar — itu sebabnya `dtype` tetap bf16.

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline

MODEL_ID = 'Wan-AI/Wan2.2-TI2V-5B-Diffusers'
dtype = torch.bfloat16

vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=dtype)
pipe.enable_model_cpu_offload()
print('model siap, dtype =', dtype)

## 6. Buat video
Setiap gambar menghasilkan satu MP4 di `/content/out`, dengan nama file yang sama seperti gambarnya.
Video yang sudah ada akan dilewati, jadi aman dijalankan ulang setelah sesi terputus.

In [ ]:
import gc, time
import numpy as np
from diffusers.utils import export_to_video, load_image

# Snap the requested box to the model's block size, keeping the image's own aspect ratio.
mod_value = pipe.vae_scale_factor_spatial * pipe.transformer.config.patch_size[1]

def fit(image, target_h, target_w):
    ratio = image.height / image.width
    h = round(np.sqrt(target_h * target_w * ratio)) // mod_value * mod_value
    w = round(np.sqrt(target_h * target_w / ratio)) // mod_value * mod_value
    return image.resize((w, h))

done = []
for i, path in enumerate(imgs):
    out_path = OUT_DIR / f'{path.stem}.mp4'
    if out_path.exists():
        print(f'[{i + 1}/{len(imgs)}] lewati {path.name} (sudah ada)')
        done.append(out_path)
        continue
    prompt = PROMPTS[i] if i < len(PROMPTS) and PROMPTS[i].strip() else DEFAULT_PROMPT
    print(f'[{i + 1}/{len(imgs)}] {path.name} -> {out_path.name}')
    image = fit(load_image(str(path)).convert('RGB'), HEIGHT, WIDTH)
    started = time.time()
    frames = pipe(
        image=image,
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=image.height,
        width=image.width,
        num_frames=NUM_FRAMES,
        guidance_scale=GUIDANCE,
        num_inference_steps=STEPS,
    ).frames[0]
    export_to_video(frames, str(out_path), fps=FPS)
    done.append(out_path)
    print(f'    selesai dalam {time.time() - started:.0f} detik')
    gc.collect()
    torch.cuda.empty_cache()

print(f'\nSelesai: {len(done)} video di {OUT_DIR}')

## 7. Unduh hasilnya
Simpan ZIP-nya, ekstrak di komputer, lalu di Kisara buka tiap klip → **Impor video** dan pilih MP4
yang sesuai. Nama file MP4 sama dengan nama gambarnya, jadi mudah dipasangkan.

In [ ]:
from google.colab import files

zip_path = shutil.make_archive('/content/kisara-videos', 'zip', OUT_DIR)
print('Ukuran:', round(os.path.getsize(zip_path) / 1e6, 1), 'MB')
files.download(zip_path)

## Kalau kehabisan memori (CUDA out of memory)

Turunkan salah satu, lalu jalankan ulang sel 5 dan 6:
- `NUM_FRAMES` dari `49` ke `33` (video lebih pendek, ~1,4 detik)
- `HEIGHT, WIDTH` dari `480, 832` ke `384, 672`
- `STEPS` dari `30` ke `20` (lebih cepat, sedikit kurang halus)

Kalau masih gagal, sesi Colab-mu mungkin sudah kena batas pemakaian — tunggu beberapa jam atau
pakai akun lain. Colab gratis membatasi pemakaian GPU secara dinamis.